# FP16-relative recovery statistics

This notebook reads existing recovery `metadata.json` files and measures only the remaining quantization damage relative to FP16. It performs no model loading or inference.

## Definition

For language $\ell$ and recovery condition $r$, define the relative change from FP16 as

$$\Delta_{\ell,r}=\frac{A_{\ell,r}-A_{\ell,\mathrm{FP16}}}{A_{\ell,\mathrm{FP16}}}\times 100\%. $$

A value closer to zero indicates less remaining quantization damage. The table reports the mean per-language relative change for Latin and non-Latin languages and the absolute difference between those group means. These three values are relative percentages, not percentage-point differences. `Std_lang` is the population standard deviation of the restored accuracies themselves across the 12 target languages, in percentage points. English is excluded.

In [1]:
from pathlib import Path
import json
import statistics
import pandas as pd

ROOT = Path(
    '/root/autodl-tmp/Multilingual_QE/result/recovery/recovery/'
    'klar/llama3.1'
)
QUANT_TYPES = ['bnb-nf4']

# Prefer exact FP16 results when available. The file must have the same
# `languages -> <lang> -> accuracy` structure as recovery metadata.
FP16_METADATA_PATH = None

# Llama 3.1 FP16 accuracies (%) from llama3.1_fp16_result.json.
FP16_ACCURACY_PERCENT = {
    'ca': 49.11808669656202, 'en': 65.45590433482795,
    'es': 54.54409566517182, 'fr': 59.85052316890879,
    'hu': 50.20926756352796, 'nl': 51.86846038863976,
    'tr': 54.03587443946197, 'ar': 47.45889387144993,
    'el': 39.58146487294474, 'he': 39.22272047832587,
    'ja': 50.74738415545598, 'ko': 40.52316890881915,
    'zh': 50.37369207772807,
}

LATIN_LANGUAGES = ['ca', 'es', 'fr', 'hu', 'nl', 'tr']
NON_LATIN_LANGUAGES = ['ar', 'el', 'he', 'ja', 'ko', 'zh']
TARGET_LANGUAGES = LATIN_LANGUAGES + NON_LATIN_LANGUAGES

PREFERRED_EXPERIMENT_ORDER = [
    'quant_baseline',
    'layer_group_0_25_attention_mlp',
    'layer_group_25_50_attention_mlp',
    'layer_group_50_75_attention_mlp',
    'layer_group_75_100_attention_mlp',
    'component_0_25_attention',
    'component_0_25_mlp',
]

# None means: automatically use every subdirectory containing metadata.json.
EXPERIMENTS = None
OUTPUT_PATH = ROOT / 'fp16_relative_recovery_statistics.csv'

In [2]:
def load_metadata(path):
    with Path(path).open('r', encoding='utf-8') as stream:
        metadata = json.load(stream)
    if not isinstance(metadata.get('languages'), dict):
        raise ValueError(f'Missing per-language accuracies in {path}')
    missing = [
        language for language in TARGET_LANGUAGES
        if language not in metadata['languages']
    ]
    if missing:
        raise ValueError(f'Missing target languages in {path}: {missing}')
    return metadata


def language_accuracies(metadata):
    return {
        language: float(metadata['languages'][language]['accuracy'])
        for language in TARGET_LANGUAGES
    }


def mean(values):
    return statistics.fmean(float(value) for value in values)


def population_std(values):
    return statistics.pstdev(float(value) for value in values)


def friendly_name(experiment_name, metadata):
    if experiment_name == 'quant_baseline':
        return 'Quant baseline'
    if metadata.get('recovery_level') == 'layer_group':
        return f"{metadata.get('percentage_range', '')} Attention+MLP".strip()
    if metadata.get('recovery_level') == 'component':
        percentage = metadata.get('percentage_range', '')
        component = str(metadata.get('component', '')).capitalize()
        return f'{percentage} {component}'.strip()
    return experiment_name


def discover_experiments(quant_dir):
    if EXPERIMENTS is not None:
        names = list(EXPERIMENTS)
    else:
        discovered = {
            child.name for child in quant_dir.iterdir()
            if child.is_dir() and (child / 'metadata.json').is_file()
        }
        names = [
            name for name in PREFERRED_EXPERIMENT_ORDER
            if name in discovered
        ]
        names.extend(sorted(discovered - set(names)))
    return names


def summarize_fp16_damage(
    quant_type, experiment_name, metadata, fp16_accuracy
):
    restored = language_accuracies(metadata)
    relative_delta_percent = {
        language: (
            (restored[language] - fp16_accuracy[language])
            / fp16_accuracy[language]
        ) * 100.0
        for language in TARGET_LANGUAGES
    }
    delta_latin = mean(
        relative_delta_percent[language]
        for language in LATIN_LANGUAGES
    )
    delta_non_latin = mean(
        relative_delta_percent[language]
        for language in NON_LATIN_LANGUAGES
    )
    return {
        'Quantization': quant_type,
        'Restored': friendly_name(experiment_name, metadata),
        'Δ Latin ↑ (%)': delta_latin,
        'Δ Non-Latin ↑ (%)': delta_non_latin,
        '|Δ L-NL| ↓ (pp)': abs(delta_latin - delta_non_latin),
        'Std_lang ↓ (pp)': 100.0 * population_std(
            restored[language] for language in TARGET_LANGUAGES
        ),
    }

In [3]:
if FP16_METADATA_PATH is not None:
    with Path(FP16_METADATA_PATH).open('r', encoding='utf-8') as stream:
        fp16_metadata = json.load(stream)
    if isinstance(fp16_metadata.get('languages'), dict):
        fp16_accuracy = language_accuracies(
            load_metadata(FP16_METADATA_PATH)
        )
    else:
        missing = [
            language for language in TARGET_LANGUAGES
            if language not in fp16_metadata
            or 'accuracy' not in fp16_metadata[language]
        ]
        if missing:
            raise ValueError(
                f'Missing FP16 per-language accuracies: {missing}'
            )
        fp16_accuracy = {
            language: float(fp16_metadata[language]['accuracy'])
            for language in TARGET_LANGUAGES
        }
    print('Using exact FP16 metadata:', FP16_METADATA_PATH)
else:
    missing = [
        language for language in TARGET_LANGUAGES
        if language not in FP16_ACCURACY_PERCENT
    ]
    if missing:
        raise ValueError(f'Missing fallback FP16 accuracies: {missing}')
    fp16_accuracy = {
        language: FP16_ACCURACY_PERCENT[language] / 100.0
        for language in TARGET_LANGUAGES
    }
    print('Using rounded FP16 accuracies from FP16_ACCURACY_PERCENT')

rows = []
for quant_type in QUANT_TYPES:
    quant_dir = ROOT / quant_type
    for experiment_name in discover_experiments(quant_dir):
        metadata_path = quant_dir / experiment_name / 'metadata.json'
        metadata = load_metadata(metadata_path)
        rows.append(summarize_fp16_damage(
            quant_type=quant_type,
            experiment_name=experiment_name,
            metadata=metadata,
            fp16_accuracy=fp16_accuracy,
        ))

results = pd.DataFrame(rows).round(2)
results.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
display(results)

Using rounded FP16 accuracies from FP16_ACCURACY_PERCENT


ValueError: Missing target languages in /root/autodl-tmp/Multilingual_QE/result/recovery/recovery/klar/llama3.1/bnb-nf4/quant_baseline/metadata.json: ['es', 'fr', 'hu', 'nl', 'tr', 'he', 'ja', 'ko', 'zh']